# El agente SQL semántico

**Lección 4 · Clase 5.4** — el capstone de la dupla. En la lección 3 construimos el mapa: un grafo en Neo4j que sabe lo que la base de datos significa. Ahora le damos ese mapa a un agente y lo ponemos a responder preguntas de negocio de verdad — con plata de por medio.

La arquitectura es deliberadamente simple: **dos herramientas y un método**.

```
                    ┌─ leer_cypher ──▶  Neo4j   (la capa semántica: qué significa)
  pregunta ──▶ 🤖 ──┤
                    └─ ejecutar_sql ─▶  SQLite  (los datos: cuánto es)
```

El método va en el system prompt y es el corazón de la lección: **DESCUBRIR** (qué tablas hablan de esto) → **VERIFICAR** (qué columnas y joins exactos) → **CALCULAR** (recién ahí, SQL) → **EXPLICAR**. Primero entender, después calcular.

| | |
|---|---|
| **El mundo** | `construir_grafo.py` reconstruye base + grafo desde cero (autocontenido, no requiere la lección 3). |
| **Las guardas** | Por qué las dos herramientas son de solo lectura, y verlas rechazar un `DROP` en vivo. |
| **El agente** | `create_agent` + `gpt-5-mini` con el método en el prompt. |
| **El contraste** | La misma pregunta con trampa a un agente **a ciegas** (solo SQL + esquema pelado) y al agente **informado**. Con la respuesta correcta calculada aparte, como juez. |
| **El recorrido** | El subgrafo que el agente caminó, dibujado con el número de paso en cada tabla. |


In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta:
# %pip install -q langchain==1.3.14 langchain-core==1.5.3 langchain-openai==1.4.2 \
#   neocarta==0.8.0 neo4j==6.2.0 pandas==2.3.3 pyyaml==6.0.3 matplotlib==3.11.1 \
#   networkx==3.6.1 python-dotenv==1.2.2
from dotenv import load_dotenv
import os

load_dotenv(override=True)  # el .env de la lección gana sobre variables heredadas del entorno

try:
    from google.colab import userdata  # type: ignore
    for llave in ("OPENAI_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"):
        try:
            os.environ[llave] = userdata.get(llave) or os.environ.get(llave, "")
        except Exception:
            pass
except Exception:
    pass

NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_AUTH = (os.environ.get("NEO4J_USERNAME", "neo4j"), os.environ.get("NEO4J_PASSWORD", "password"))
NEO4J_DB = os.environ.get("NEO4J_DATABASE", "neo4j")

import logging

logging.getLogger("neo4j").setLevel(logging.ERROR)  # sin avisos GQL en los traces

from neo4j import GraphDatabase


def hay_neo4j() -> bool:
    try:
        with GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH, connection_timeout=3.0) as driver:
            driver.verify_connectivity()
        return True
    except Exception:
        return False


HAY_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
HAY_NEO4J = hay_neo4j()
print("OPENAI_API_KEY presente:", HAY_OPENAI)
print(f"Neo4j accesible en {NEO4J_URI}:", HAY_NEO4J)
if not (HAY_OPENAI and HAY_NEO4J):
    print("⚠️ Esta lección necesita ambos: sin alguno, las celdas del agente se saltan")
    print("   (la base SQLite y la respuesta correcta con pandas corren igual).")

import urllib.request
from pathlib import Path

BASE_RAW = (
    "https://raw.githubusercontent.com/josepenam/clases-diplomado-gen-ia/main/"
    "class_5_4_integraciones/leccion4_agente_sql_semantico/"
)


def asegurar(nombre: str) -> Path:
    ruta = Path(nombre)
    if not ruta.exists():
        pedido = urllib.request.Request(
            BASE_RAW + nombre, headers={"User-Agent": "Mozilla/5.0 (clase-diplomado-gen-ia)"}
        )
        ruta.write_bytes(urllib.request.urlopen(pedido).read())
    return ruta


for archivo in (
    "crear_base_datos.py", "generar_metadata_csv.py",
    "construir_capa_negocio.py", "construir_grafo.py", "ontologia.yaml",
):
    asegurar(archivo)

Path("outputs").mkdir(exist_ok=True)

## Reconstruir el mundo

Autocontención: [`construir_grafo.py`](construir_grafo.py) crea la base mock, genera la metadata, **borra el grafo** y lo reconstruye completo (capa técnica de neocarta + capa de negocio de la ontología). Correrlo dos veces da exactamente lo mismo.

> El script tiene una guardia: solo borra grafos vacíos o construidos por esta clase. Si tu Neo4j tiene otra cosa adentro, aborta y te pide un contenedor dedicado.

In [ ]:
import crear_base_datos

if HAY_NEO4J:
    import construir_grafo

    construir_grafo.construir()
else:
    crear_base_datos.crear(verboso=False)
    print("Sin Neo4j: creada solo la base SQLite.")

## Las dos herramientas, con sus guardas

Un agente con acceso a una base de datos es un agente que puede romper una base de datos — salvo que sea *estructuralmente incapaz*. Las dos herramientas son de solo lectura por diseño, con **dos cinturones cada una**:

- `leer_cypher` corre dentro de `execute_read` de Neo4j: una transacción de lectura donde cualquier escritura falla, decida lo que decida el modelo.
- `ejecutar_sql` valida el texto (solo `SELECT`/`WITH`, sin multi-statement, lista de keywords prohibidas) **y además** abre SQLite en modo solo-lectura (`mode=ro`) — si la validación fallara, el sistema de archivos no coopera.

La lección de diseño: las guardas van en la *herramienta*, no en el prompt. Pedirle "por favor no borres nada" a un modelo es una sugerencia; `mode=ro` es una ley de la física.

In [ ]:
import json as json_lib
import re
import sqlite3

from langchain_core.tools import tool

MAX_FILAS_CYPHER = 100
MAX_FILAS_SQL = 200

_PROHIBIDAS = re.compile(
    r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|ALTER|TRUNCATE|GRANT|REVOKE|CREATE"
    r"|ATTACH|DETACH|PRAGMA|VACUUM|REINDEX|BEGIN|COMMIT)\b",
    re.IGNORECASE,
)


def validar_sql(sql: str) -> str | None:
    """Devuelve el motivo del rechazo, o None si la consulta es aceptable."""
    cuerpo = re.sub(r"--[^\n]*|/\*.*?\*/", " ", sql, flags=re.S).strip().rstrip(";").strip()
    if not cuerpo:
        return "consulta vacía"
    primera = cuerpo.split(None, 1)[0].upper()
    if primera not in ("SELECT", "WITH"):
        return f"solo se permiten SELECT/WITH (llegó {primera})"
    if ";" in cuerpo:
        return "no se permiten múltiples sentencias"
    if m := _PROHIBIDAS.search(cuerpo):
        return f"keyword prohibida: {m.group(0).upper()}"
    return None


driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH) if HAY_NEO4J else None


@tool
def leer_cypher(consulta: str) -> str:
    """Consulta de SOLO LECTURA (Cypher) sobre la capa semántica en Neo4j.

    El grafo describe la base de datos montania: (:Table)-[:HAS_COLUMN]->(:Column),
    dominios (:Table)-[:IN_DOMAIN]->(:BusinessDomain), métricas
    (:Table)-[:ABOUT]->(:Metric) y (:Column)-[:MEASURES]->(:Metric), monedas
    (:Column)-[:IN_CURRENCY]->(:Currency), y joins válidos
    (:Column)-[r:REFERENCES]->(:Column) donde r.criteria trae la condición lista
    para SQL. Las tablas tienen t.descripcion en español.

    Recetas — cópialas y ajusta en vez de inventar Cypher:
    1. Mapa completo (empieza SIEMPRE aquí; evita adivinar nombres):
       MATCH (t:Table) OPTIONAL MATCH (t)-[:ABOUT]->(m:Metric)
       RETURN t.name, t.dominio, t.descripcion, collect(m.name) AS metricas
    2. Columnas de una tabla, con moneda y qué miden:
       MATCH (t:Table {name:'trx_pos'})-[:HAS_COLUMN]->(c:Column)
       OPTIONAL MATCH (c)-[:IN_CURRENCY]->(cur:Currency)
       OPTIONAL MATCH (c)-[:MEASURES]->(m:Metric)
       RETURN c.name, c.descripcion, cur.code AS moneda, m.name AS mide
    3. Joins válidos que tocan una tabla:
       MATCH (t1:Table)-[:HAS_COLUMN]->()-[r:REFERENCES]-()<-[:HAS_COLUMN]-(t2:Table)
       WHERE t1.name = 'trx_pos'
       RETURN DISTINCT t2.name AS con_tabla, r.criteria AS criteria

    Reglas del dialecto (errores típicos):
    - Los patrones (t)-[:X]->() van SOLO en MATCH/OPTIONAL MATCH; en RETURN o
      WHERE producen SyntaxError (usa EXISTS { MATCH ... } si necesitas filtrar).
    - No existe el tipo comodín [:?]; varios tipos se escriben [:A|B], solo en MATCH.
    - Usa solo los nombres de tabla que devolvió la receta 1 — no los inventes.
    """

    def _leer(tx):
        return [registro.data() for registro in tx.run(consulta)][:MAX_FILAS_CYPHER]

    try:
        with driver.session(database=NEO4J_DB) as sesion:
            filas = sesion.execute_read(_leer)
        return json_lib.dumps(filas, ensure_ascii=False, default=str) or "[]"
    except Exception as error:
        return f"ERROR Cypher: {error}"


@tool
def ejecutar_sql(consulta: str) -> str:
    """Consulta SQL de SOLO LECTURA (SELECT/WITH) sobre la base SQLite montania.

    Dialecto SQLite. Usa siempre LIMIT. Las fechas son texto ISO (usa
    strftime('%m', ts) para filtrar por mes). UNA sola sentencia, sin ';'.
    PRAGMA está bloqueado — el DDL de una tabla se consulta así:
    SELECT sql FROM sqlite_master WHERE type='table' AND name='trx_pos'
    """
    if motivo := validar_sql(consulta):
        return f"RECHAZADA: {motivo}"
    try:
        with sqlite3.connect("file:outputs/montania.db?mode=ro", uri=True) as conexion:
            cursor = conexion.execute(consulta)
            columnas = [d[0] for d in cursor.description]
            filas = cursor.fetchmany(MAX_FILAS_SQL)
        return json_lib.dumps([dict(zip(columnas, fila)) for fila in filas], ensure_ascii=False)
    except Exception as error:
        return f"ERROR SQL: {error}"


# Las guardas, en vivo — esto es lo que le pasa a un agente que intenta escribir:
print(ejecutar_sql.invoke({"consulta": "DROP TABLE trx_pos"}))
print(ejecutar_sql.invoke({"consulta": "SELECT 1; DELETE FROM cli"}))
if HAY_NEO4J:
    print(leer_cypher.invoke({"consulta": "MERGE (h:Hacker {yo: 'estuve aquí'})"})[:120])

## El método, en el prompt

El system prompt codifica el flujo de trabajo de un analista senior. No es prosa decorativa: cada regla existe porque sin ella el agente falla de una forma específica que ya vimos (adivina joins, suma monedas distintas, no declara el período). Y el método tiene dos mitades: el *flujo* va en el system prompt, pero las *recetas* de consulta y las reglas del dialecto van en el docstring de cada herramienta — para el modelo, la descripción de la herramienta es su página de manual.

In [ ]:
INSTRUCCIONES = """Eres el analista de datos senior de un centro de esquí chileno. Respondes
preguntas de negocio PRIMERO entendiendo la estructura en la capa semántica
(Neo4j), DESPUÉS calculando con SQL real, y al final explicando los números.

Método — síguelo en orden:
1. DESCUBRIR: usa leer_cypher empezando por la receta 1 del docstring (el mapa
   completo de tablas, dominios y métricas). De ahí elige las candidatas por
   t.descripcion — nunca adivines nombres de tablas.
2. VERIFICAR: antes de escribir SQL, consulta las columnas exactas, qué miden,
   EN QUÉ MONEDA ([:IN_CURRENCY]) y los joins válidos ([:REFERENCES], usa
   r.criteria como condición).
3. CALCULAR: SQL de solo lectura sobre SQLite con ejecutar_sql. Agrega en SQL
   (no sumes de cabeza), filtra temprano y usa LIMIT.
4. EXPLICAR: entrega los números con las tablas usadas, el período cubierto y
   los supuestos.

Reglas duras:
- Montos en monedas distintas JAMÁS se suman ni comparan directo: convierte
  (la capa semántica te dice con qué) o preséntalos por separado, etiquetados.
- Declara siempre el período que usaste.
- Si una consulta falla, lee el error y corrígela.
- No explores de más: el mapa (receta 1) + 2–3 consultas de las recetas basta
  para pasar a CALCULAR. No repitas consultas que ya hiciste ni tantees el
  esquema por SQL: esa información vive en el grafo.
Responde en español.
"""

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

MODELO = "gpt-5-mini"

if HAY_OPENAI and HAY_NEO4J:
    agente_informado = create_agent(
        ChatOpenAI(model=MODELO),
        tools=[leer_cypher, ejecutar_sql],
        system_prompt=INSTRUCCIONES,
    )
    print("Agente informado listo: leer_cypher + ejecutar_sql + el método.")

## Verlo trabajar

Una pregunta de calentamiento que exige juntar tres tablas (`trx_pos → cat_sku → lin_prod`). Del trace nos interesa el *orden*: primero Cypher (entender), después SQL (calcular).

In [ ]:
from langgraph.errors import GraphRecursionError


def correr(agente, pregunta: str, limite_pasos: int = 80, reintentos: int = 1):
    """Invoca al agente y devuelve (respuesta, pasos) donde pasos es
    [(n, herramienta, argumentos, resultado)]. Los agentes no son deterministas:
    si uno se enreda y agota los pasos, reintentamos una vez."""
    try:
        resultado = agente.invoke(
            {"messages": [{"role": "user", "content": pregunta}]},
            config={"recursion_limit": limite_pasos},
        )
    except GraphRecursionError:
        if reintentos > 0:
            print(f"  (el agente agotó {limite_pasos} pasos; reintentando…)")
            return correr(agente, pregunta, limite_pasos, reintentos - 1)
        return f"(el agente no convergió en {limite_pasos} pasos)", []
    pasos, resultados_tool = [], {}
    for mensaje in resultado["messages"]:
        if type(mensaje).__name__ == "ToolMessage":
            resultados_tool[mensaje.tool_call_id] = str(mensaje.content)
    n = 0
    for mensaje in resultado["messages"]:
        for llamada in getattr(mensaje, "tool_calls", None) or []:
            n += 1
            pasos.append((n, llamada["name"], llamada["args"],
                          resultados_tool.get(llamada["id"], "")))
    return resultado["messages"][-1].content, pasos


def mostrar_trace(pasos, respuesta, ancho: int = 100):
    for n, herramienta, argumentos, resultado in pasos:
        consulta = str(argumentos.get("consulta", argumentos))[:ancho]
        print(f"  [{n}] {herramienta}: {consulta}")
        print(f"       → {resultado[:ancho]}")
    print(f"\n🤖 {respuesta}")


if HAY_OPENAI and HAY_NEO4J:
    respuesta, pasos = correr(agente_informado, "¿Cuánto facturamos por línea de producto en julio?")
    mostrar_trace(pasos, respuesta)

## El contraste: la pregunta con trampa

Ahora el experimento central. La pregunta suena inocente:

> *"¿De cuánto fue la facturación total de julio, incluyendo las reservas de agencias, en pesos chilenos?"*

La trampa la conocemos de la lección 3: `res_agt.imp` está en pesos **argentinos**, con montos del mismo orden de magnitud que los chilenos, y nada en el esquema lo dice (la tabla de tipo de cambio se llama `tc_d` y su columna, `vlr` — suerte adivinando). Primero calculemos la respuesta correcta a mano —con pandas, mirando la ontología— para tener un juez imparcial:

In [ ]:
import pandas as pd

conexion = sqlite3.connect("file:outputs/montania.db?mode=ro", uri=True)

pos_julio = pd.read_sql(
    "SELECT SUM(mnt) AS total FROM trx_pos WHERE strftime('%m', ts) = '07'", conexion
)["total"][0]

agencias_julio_clp = pd.read_sql(
    """SELECT SUM(r.imp * tc.vlr) AS total
       FROM res_agt r JOIN tc_d tc ON r.f_res = tc.f
       WHERE strftime('%m', r.f_res) = '07'""",
    conexion,
)["total"][0]

agencias_julio_ars = pd.read_sql(
    "SELECT SUM(imp) AS total FROM res_agt WHERE strftime('%m', f_res) = '07'", conexion
)["total"][0]

CORRECTO = pos_julio + agencias_julio_clp
INCORRECTO_SUMA_DIRECTA = pos_julio + agencias_julio_ars

print(f"POS julio (CLP):                      {pos_julio:>18,.0f}")
print(f"Agencias julio (ARS):                 {agencias_julio_ars:>18,.0f}")
print(f"Agencias julio convertidas (CLP):     {agencias_julio_clp:>18,.0f}")
print(f"────────────────────────────────────────────────────────")
print(f"✓ CORRECTO (POS + agencias en CLP):   {CORRECTO:>18,.0f}")
print(f"✗ el número que NO existe (CLP+ARS):  {INCORRECTO_SUMA_DIRECTA:>18,.0f}")

El **agente a ciegas** es el text-to-SQL típico: mismo modelo, misma herramienta SQL, pero en vez de capa semántica recibe lo que cualquier tutorial le pega al prompt — el esquema pelado.

In [ ]:
if HAY_OPENAI:
    esquema_pelado = "\n".join(
        f"{tabla}({', '.join(col[1] for col in conexion.execute(f'PRAGMA table_info({tabla})'))})"
        for (tabla,) in conexion.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    )

    agente_ciego = create_agent(
        ChatOpenAI(model=MODELO),
        tools=[ejecutar_sql],
        system_prompt=(
            "Eres un analista de datos. Respondes preguntas consultando la base "
            "SQLite montania con la herramienta ejecutar_sql (usa LIMIT, fechas ISO texto). "
            "Este es el esquema:\n\n" + esquema_pelado + "\n\nResponde en español."
        ),
    )
    print("Agente a ciegas listo: mismo modelo, solo SQL + esquema pelado.")

In [ ]:
PREGUNTA_TRAMPA = (
    "¿De cuánto fue la facturación total de julio, "
    "incluyendo las reservas de agencias, en pesos chilenos?"
)


def parsear_numeros(texto: str) -> set[float]:
    """Números grandes del texto, tolerante a separadores chilenos y gringos."""
    candidatos = set()
    for pedazo in re.findall(r"\d[\d.,]{4,}\d", texto):
        pelado = re.sub(r"[.,]", "", pedazo)
        try:
            n = float(pelado)
            candidatos.update({n, n / 100})  # /100 por si absorbimos dos decimales
        except ValueError:
            pass
    return candidatos


def evaluar(nombre: str, respuesta: str, pasos) -> dict:
    numeros = parsear_numeros(respuesta)
    acierta = any(abs(n - CORRECTO) / CORRECTO < 0.02 for n in numeros)
    # tolerancia fina: el número equivocado está a solo ~0,2% del subtotal POS legítimo
    peras_con_manzanas = any(abs(n - INCORRECTO_SUMA_DIRECTA) / INCORRECTO_SUMA_DIRECTA < 0.01 for n in numeros)
    return {
        "agente": nombre,
        "llamadas Cypher": sum(1 for _, h, _, _ in pasos if h == "leer_cypher"),
        "llamadas SQL": sum(1 for _, h, _, _ in pasos if h == "ejecutar_sql"),
        "¿acierta? (±2%)": "✓" if acierta else "✗",
        "¿sumó CLP+ARS?": "⚠️ SÍ" if peras_con_manzanas else "no",
    }


if HAY_OPENAI and HAY_NEO4J:
    veredictos = []
    for nombre, agente in [("a ciegas", agente_ciego), ("informado", agente_informado)]:
        respuesta, pasos = correr(agente, PREGUNTA_TRAMPA)
        veredictos.append(evaluar(nombre, respuesta, pasos))
        print(f"═══ Agente {nombre} ═══")
        mostrar_trace(pasos, respuesta)
        print()
        if nombre == "informado":
            pasos_informado = pasos  # para el dibujo del recorrido

    display(pd.DataFrame(veredictos).set_index("agente"))

La tabla de veredictos es la lección entera: mismo modelo, misma pregunta, misma herramienta SQL. La única diferencia es si el agente puede *consultar el significado* antes de calcular. El agente a ciegas no tiene forma de saber que `imp` está en pesos argentinos — los montos parecen chilenos perfectamente normales, así que los suma o los ignora. El informado lo descubre en el grafo (`IN_CURRENCY` → ARS), encuentra `tc_d` por los joins declarados, y convierte.

> Si corres esto varias veces verás variar los caminos (los agentes no son deterministas) — lo que no varía es la asimetría de información: uno *puede* saber, el otro no.

## El recorrido, dibujado

Cierre visual: el subgrafo de la capa semántica que el agente informado efectivamente caminó — qué tablas tocó (con el número de paso) y a qué conceptos están conectadas.

In [ ]:
if HAY_OPENAI and HAY_NEO4J:
    import matplotlib.pyplot as plt
    import networkx as nx

    TABLAS = [fila[0] for fila in conexion.execute("SELECT name FROM sqlite_master WHERE type='table'")]

    def tablas_tocadas(pasos) -> dict[str, list[int]]:
        """tabla → pasos cuyo input o resultado la mencionan (por palabra completa)."""
        tocadas: dict[str, list[int]] = {}
        for n, _, argumentos, resultado in pasos:
            texto = json_lib.dumps(argumentos, default=str) + " " + resultado
            for tabla in TABLAS:
                if re.search(rf"\b{tabla}\b", texto):
                    tocadas.setdefault(tabla, []).append(n)
        return tocadas

    tocadas = tablas_tocadas(pasos_informado)

    with driver.session(database=NEO4J_DB) as sesion:
        conceptos = [registro.data() for registro in sesion.run("""
            MATCH (t:Table) WHERE t.name IN $nombres
            OPTIONAL MATCH (t)-[r:IN_DOMAIN|ABOUT]-(concepto)
            RETURN t.name AS tabla, t.dominio AS dominio, type(r) AS rel,
                   labels(concepto)[0] AS etiqueta, concepto.name AS nombre
        """, nombres=list(tocadas))]
        joins = [registro.data() for registro in sesion.run("""
            MATCH (t1:Table)-[:HAS_COLUMN]->(:Column)-[:REFERENCES]->(:Column)<-[:HAS_COLUMN]-(t2:Table)
            WHERE t1.name IN $nombres AND t2.name IN $nombres AND t1 <> t2
            RETURN DISTINCT t1.name AS a, t2.name AS b
        """, nombres=list(tocadas))]

    G = nx.Graph()
    dominio_de = {}
    for fila in conceptos:
        G.add_node(fila["tabla"], tipo="tabla")
        dominio_de[fila["tabla"]] = fila["dominio"]
        if fila["nombre"] and fila["etiqueta"] in ("BusinessDomain", "Metric", "Currency"):
            clave = f"{fila['etiqueta']}:{fila['nombre']}"
            G.add_node(clave, tipo="concepto", display=fila["nombre"])
            G.add_edge(fila["tabla"], clave, rel=fila["rel"])
    for join in joins:
        G.add_edge(join["a"], join["b"], rel="JOIN")

    colores_dominio = {"ventas": "#2a9d8f", "montania": "#457b9d", "clientes": "#e9c46a"}
    colores, tamanos, etiquetas = [], [], {}
    for nodo, datos in G.nodes(data=True):
        if datos["tipo"] == "tabla":
            colores.append(colores_dominio.get(dominio_de.get(nodo), "#999999"))
            tamanos.append(1500)
            numeros = "·".join(map(str, sorted(set(tocadas.get(nodo, [])))))
            etiquetas[nodo] = f"{nodo}\n[paso {numeros}]"
        else:
            colores.append("#e76f51")
            tamanos.append(700)
            etiquetas[nodo] = datos["display"]

    plt.figure(figsize=(11, 7))
    posiciones = nx.spring_layout(G, seed=54, k=1.2)
    nx.draw_networkx_nodes(G, posiciones, node_color=colores, node_size=tamanos, alpha=0.92)
    nx.draw_networkx_edges(G, posiciones, alpha=0.4)
    nx.draw_networkx_edge_labels(
        G, posiciones,
        {(a, b): d["rel"] for a, b, d in G.edges(data=True) if d.get("rel")},
        font_size=7,
    )
    nx.draw_networkx_labels(G, posiciones, etiquetas, font_size=8)
    plt.title(f"El recorrido del agente informado — {PREGUNTA_TRAMPA[:70]}…")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig("outputs/recorrido_agente.png", dpi=110)
    plt.show()

## Cierre

Lo que acabas de ver escala — y lo que no, también conviene saberlo:

- **Escala**: el patrón dos-herramientas-un-método es el mismo con BigQuery, Postgres o Snowflake en vez de SQLite, y con cientos de tablas en vez de 11. De hecho, esta lección es la adaptación docente de un agente real sobre un data lake corporativo. Con muchas tablas, la búsqueda en el grafo pasa de `CONTAINS` a búsqueda vectorial — `neocarta[mcp]` expone exactamente eso como servidor MCP (lección 1 + lección 3, juntas).
- **No escala solo**: la ontología es mantenimiento. Alguien tiene que declarar que `imp` está en USD *antes* de que el agente lo necesite. La capa semántica es un producto de datos con dueños — el agente es la parte fácil.
- **La seguridad no fue opcional**: guardas estructurales (transacción de lectura, `mode=ro`, validación) en cada herramienta. Un agente de producción suma autenticación, cuotas y auditoría de cada consulta.


In [ ]:
if driver:
    driver.close()
conexion.close()
print("Fin del capstone semántico.")